In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
## Imports the specific WordCloud class from the wordcloud library to visualize word frequencies.
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

In [17]:
# Imports the 'stopwords' module from NLTK, which contains a list of common words (e.g., 'the', 'is', 'a').
# Downloading NLTK data
print(nltk.download('stopwords'))   # Downloads the list of stopwords.
print(nltk.download('punkt'))       # Downloads the 'punkt' tokenizer models, used for splitting text into words/sentences.
print(nltk.download('punkt_tab'))   # Downloads the 'punkt_tab' resource needed for tokenization.

True
True
True


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
df = pd.read_csv('/content/drive/MyDrive/spam.csv')

In [7]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [8]:
df.drop(columns = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace = True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
df.rename(columns = {'v1': 'target', 'v2': 'text'}, inplace = True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [10]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
df.duplicated().sum()

np.int64(403)

In [12]:
len(df)

5572

In [13]:
df = df.drop_duplicates(keep = 'first')
len(df)

5169

In [14]:
from nltk.stem.porter import PorterStemmer
import string
ps = PorterStemmer()

def transform_text(text):
  text=text.lower()
  text = nltk.word_tokenize(text)
  y = []
  for i in text:
    if i.isalnum():
      y.append(i)
  text = y[:]
  y.clear()
  for i in text:
    if i not in stopwords.words('english') and i not in string.punctuation:
      y.append(i)
  text = y[:]
  y.clear()
  for i in text:
    y.append(ps.stem(i))
  return " ".join(y)

In [19]:
import nltk
try:
    nltk.data.find('tokenizers/punkt_tab')
    print('punkt_tab is already available.')
except nltk.downloader.DownloadError:
    print('Error downloading punkt_tab.')
except LookupError:
    print('punkt_tab not found. Downloading now...')
    nltk.download('punkt_tab')
    print('punkt_tab download attempted. Please try running the text transformation cell again.')

punkt_tab is already available.


In [20]:
transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [21]:
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [23]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfidf = TfidfVectorizer(max_features = 500)
# Creates an instance of TfidfVectorizer.
# 'max_features=500' limits the vocabulary to the 500 most frequent words in the dataset. This saves memory and can prevent overfitting.

In [24]:
X = tfidf.fit_transform(df['transformed_text']).toarray()
# 'fit_transform' does two things:
# 1. 'fit': It learns the 500-word vocabulary from all documents in 'transformed_text'.
# 2. 'transform': It converts each document into a numerical vector of length 500 based on the TF-IDF scores of those words.
# '.toarray()' converts the default sparse matrix into a dense NumPy array.
# 'X' is now our feature matrix (input for the models).
y = df['target'].values
# 'y' is our target vector (output). '.values' extracts the 'target' column as a NumPy array.

In [25]:
from sklearn.model_selection import train_test_split
X_train, X_test , y_train, y_test = train_test_split(X,y,test_size = 0.20, random_state = 2)

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
# Imports all the classifier classes we want to test.

svc = SVC(kernel= "sigmoid", gamma  = 1.0) # Support Vector Machine
knc = KNeighborsClassifier() # K-Nearest Neighbors
mnb = MultinomialNB() # Multinomial Naive Bayes (often good for text)
dtc = DecisionTreeClassifier(max_depth = 5) # Decision Tree (limited depth to prevent overfitting)
lrc = LogisticRegression(solver = 'liblinear', penalty = 'l1') # Logistic Regression
rfc = RandomForestClassifier(n_estimators = 50, random_state = 2 ) # Random Forest
abc = AdaBoostClassifier(n_estimators = 50, random_state = 2) # AdaBoost
bc = BaggingClassifier(n_estimators = 50, random_state = 2) # Bagging
etc = ExtraTreesClassifier(n_estimators = 50, random_state = 2) # Extra Trees
gbdt = GradientBoostingClassifier(n_estimators = 50, random_state = 2) # Gradient Boosting
xgb  = XGBClassifier(n_estimators = 50, random_state = 2) # XGBoost
# Initializes one instance of each classifier, setting some basic parameters to control complexity.

clfs = {
    'SVC': svc,
    'KNN': knc,
    'NB': mnb,
    'DT': dtc,
    'LR': lrc,
    'RF': rfc,
    'Adaboost': abc,
    'Bgc': bc,
    'ETC': etc,
    'GBDT': gbdt,
    'xgb': xgb
    }
# Creates a dictionary 'clfs' that maps a string name (e.g., 'SVC') to the corresponding model object (e.g., the 'svc' instance).
# This makes it easy to loop through and train/evaluate all of them automatically.

In [27]:
from sklearn.metrics import accuracy_score, precision_score
# Imports the 'accuracy_score' and 'precision_score' functions for evaluation.
# Accuracy = (Correct Predictions) / (Total Predictions)
# Precision = (True Positives) / (True Positives + False Positives). Crucial for spam: "Of all messages we flagged as spam, how many actually were?"

def train_classifier(clfs, X_train, y_train, X_test, y_test):
    # Defines a function that takes a classifier ('clfs') and the training/testing data.
    clfs.fit(X_train,y_train)
    # Trains (fits) the classifier on the training data.
    y_pred = clfs.predict(X_test)
    # Uses the trained model to make predictions on the unseen test features ('X_test').
    accuracy = accuracy_score(y_test, y_pred)
    # Calculates the accuracy by comparing the model's predictions ('y_pred') to the true labels ('y_test').
    precision = precision_score(y_test, y_pred)
    # Calculates the precision.
    return accuracy , precision
    # Returns the two calculated scores.

accuracy_scores = []
precision_scores = []
# Initializes two empty lists to store the scores from each model.

for name , clfs in clfs.items():
    # Loops through the 'clfs' dictionary, getting the 'name' (string) and 'clfs' (model object) for each item.

    current_accuracy, current_precision = train_classifier(clfs, X_train, y_train, X_test, y_test)
    # Calls the 'train_classifier' function on the current model and stores its scores.

    print()
    # Prints a blank line for spacing.
    print("For: ", name)
    # Prints the name of the model being evaluated.
    print("Accuracy: ", current_accuracy)
    # Prints its accuracy.
    print("Precision: ", current_precision)
    # Prints its precision.

    accuracy_scores.append(current_accuracy)
    # Adds the model's accuracy to the 'accuracy_scores' list.
    precision_scores.append(current_precision)
    # Adds the model's precision to the 'precision_scores' list.


For:  SVC
Accuracy:  0.9671179883945842
Precision:  0.9333333333333333

For:  KNN
Accuracy:  0.9274661508704062
Precision:  1.0

For:  NB
Accuracy:  0.9709864603481625
Precision:  0.9655172413793104

For:  DT
Accuracy:  0.9390715667311412
Precision:  0.9120879120879121

For:  LR
Accuracy:  0.9622823984526112
Precision:  0.9541284403669725

For:  RF
Accuracy:  0.9700193423597679
Precision:  0.9421487603305785

For:  Adaboost
Accuracy:  0.9235976789168279
Precision:  0.8734177215189873

For:  Bgc
Accuracy:  0.9622823984526112
Precision:  0.9024390243902439

For:  ETC
Accuracy:  0.9709864603481625
Precision:  0.921875

For:  GBDT
Accuracy:  0.9497098646034816
Precision:  0.93

For:  xgb
Accuracy:  0.9690522243713733
Precision:  0.9568965517241379
